In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
ARIZE_SPACE_ID = os.getenv("ARIZE_SPACE_ID")
ARIZE_API_KEY = os.getenv("ARIZE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Tracing System 
What type of tool are they used,
How AI think,
Step by Step working, 

In [3]:
from arize.otel import register # Connection with Arize
from openinference.instrumentation.openai import OpenAIInstrumentor #Track call record of openai
from openinference.instrumentation.agno import AgnoInstrumentor #Track the action of agno agent
import httpx 

model_id="TRAVEL_AGENT_DEMO"
tracer_provider=register(
    space_id=os.getenv("ARIZE_SPACE_ID"),
    api_key=os.getenv("ARIZE_API_KEY"),
    project_name=model_id,
    set_global_tracer_provider=True
)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
AgnoInstrumentor().instrument(tracer_provider=tracer_provider)


OpenTelemetry Tracing Details
|  Arize Project: TRAVEL_AGENT_DEMO
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [4]:
from opentelemetry import trace
tracer=trace.get_tracer(__name__)

In [5]:
def _search_api(query: str) -> str | None:
    """Try tavily search first, fall back to none"""
    tavily_key=os.getenv("TAVILY_API_KEY")
    if not tavily_key:
        return None
    try:
        resp=httpx.post(
            "https://api.tavily.com/search",
            json={
                "api_key":tavily_key,
                "query":query,
                "max_result":3,
                "search_depth":"basic",
                "include_answer":True,
            },
            timeout=8,
        )
        data=resp.json()
        answer=data.get("answer") or ""
        snippets=[r.get("content","") for r in data.get("result",[])]
        combined=" ".join([answer] + snippets).strip()
        return combined[:400] if combined else None
    except Exception:
        return None

def _compact(text: str, limit: int = 200)-> str:
    cleaned=" ".join(text.split())
    if len(cleaned) <= limit:
        return cleaned
    else:
        return cleaned[:limit].rsplit(" ", 1)[0]
        


In [6]:
from agno.tools import tool
@tool
def essential_info(destination: str)->str:
    """Get basic travle info (weather, best time, attractions, etiquette)"""
    q=f"{destination} travel essentials weather best time top attraction etiquette"
    s=_search_api(query=q)
    if s:
        return f"{destination} essentials: {_compact(s)}"
    return f"{destination} is a popular travel destination. Expect local culture, cuisine, and landmarkworth exploring."

@tool
def budget_basic(destination: str, duration: str)-> str:
    """Summerize travel cost categories"""
    q=f"{destination} travel budget average daily cost {duration}"
    s=_search_api(query=q)
    if s:
        return f"{destination} budget ({duration}): {_compact(s)}"
    return f"Budget for {duration} in {destination} depends on lodging, meals, transport, and attractions"

@tool
def local_flavor(destination: str, interests: str = "local culture")-> str:
    """Suggest authentic local experiences""" 
    q=f"{destination} authentics local experiences {interests}"
    s=_search_api(q)
    if s:
        return f"{destination} {interests}: {_compact(s)}"
    return f"Explore {destination}'s unique {interests} through market, neighborhoods and local eateries."


# Create Agent

In [7]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

trip_agent=Agent(
    name="TripPlanner",
    role="AI Travel Assistant",
    model=OpenAIChat(id="gpt-4o-mini"),
    instructions=(
        "you are a friendly and knowladgeable travel planner"
        "combine multiple tool to create trip plan including essentials, budget, and local flavour"
        "keep the tone natural, clear and under 1000 words"
    ),
    markdown=True,
    tools=[essential_info, budget_basic, local_flavor]

)




# Run Agent

In [8]:
destination="tokyo"
duration="5 days"
interest="food, culture"

query=f"""
plan a {duration} trip to {destination}.
focus on {interest}
include essential info, budget breakdown, and local experiences"""

trip_agent.print_response(
    query,
    stream=True
)

d:\Arize\myvenv\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\Arize\myvenv\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\Arize\myvenv\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\Arize\myvenv\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\Arize\myvenv\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

d:\Arize\myvenv\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\Arize\myvenv\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\Arize\myvenv\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\Arize\myvenv\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

d:\Arize\myvenv\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')